!pip install transformers datasets scikit-learn -q


In [3]:
from datasets import load_dataset

dataset = load_dataset("descartes100/enhanced-financial-phrasebank")
print(dataset)
print(dataset[list(dataset.keys())[0]][0])

README.md:   0%|          | 0.00/346 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/794k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4846 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['train'],
        num_rows: 4846
    })
})
{'train': {'label': 1, 'sentence': "Amidst the company's rapid growth in Russia, Gran confirmed that there are no immediate plans to shift all production to the region. Despite its promising prospects in Russia, the company seems to be cautious about relocating production entirely."}}


In [7]:
# The dataset stores each row nested as {'train': {'label':..., 'sentence':...}}
# Flatten it so 'label' and 'sentence' are top-level columns.
flat = dataset["train"].map(lambda row: row["train"])
flat = flat.remove_columns(["train"])

dataset = flat.train_test_split(test_size=0.2, seed=42)
print(dataset)
print(dataset["train"][0])
print("Label meanings: 0 = negative, 1 = neutral, 2 = positive")

Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'sentence'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['label', 'sentence'],
        num_rows: 776
    })
})
{'label': 0, 'sentence': 'Despite obtaining Tikkurila shares in March 2010, Solidium faces a significant capital loss of EUR5m from the upcoming sale. This unfortunate outcome may raise concerns about their investment strategies and financial performance, casting a negative shadow on their near-term prospects in the market.'}
Label meanings: 0 = negative, 1 = neutral, 2 = positive


In [8]:
from transformers import AutoTokenizer

model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)
print(tokenized)

Map:   0%|          | 0/3100 [00:00<?, ? examples/s]

Map:   0%|          | 0/776 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'sentence', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['label', 'sentence', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 776
    })
})


In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [10]:
training_args = TrainingArguments(
    output_dir="./finbert_finetuned",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.546367,0.446681,0.818299,0.818877
2,0.340109,0.424921,0.824742,0.826589
3,0.194107,0.458802,0.831186,0.831168


TrainOutput(global_step=582, training_loss=0.4373971431730539, metrics={'train_runtime': 238.101, 'train_samples_per_second': 39.059, 'train_steps_per_second': 2.444, 'total_flos': 611738696217600.0, 'train_loss': 0.4373971431730539, 'epoch': 3.0})

In [11]:
results = trainer.evaluate()
print(results)
print(f"\nFinal F1 score: {results['eval_f1']*100:.1f}%")
print(f"Final accuracy: {results['eval_accuracy']*100:.1f}%")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.194107,0.458802,3,0.831186,0.831168


{'eval_loss': 0.4588022828102112, 'eval_accuracy': 0.8311855670103093, 'eval_f1': 0.8311679504771936}

Final F1 score: 83.1%
Final accuracy: 83.1%


In [12]:
trainer.save_model("./finbert_finetuned_final")
tokenizer.save_pretrained("./finbert_finetuned_final")
print("Saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


In [13]:
import shutil
shutil.make_archive("finbert_finetuned_final", "zip", "./finbert_finetuned_final")
from google.colab import files
files.download("finbert_finetuned_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>